In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

PROJECT_ROOT = Path.cwd()
TRAIN_PATH = PROJECT_ROOT / "data" / "processed" / "splits" / "train.parquet"
MOVIES_PATH = PROJECT_ROOT / "data" / "raw" / "ml-20m" / "movies.csv"

In [2]:
train_df = pd.read_parquet(TRAIN_PATH, columns=["userId", "movieId", "rating"])

user_ids = train_df["userId"].unique()
movie_ids = train_df["movieId"].unique()

user_to_idx = {uid: idx for idx, uid in enumerate(user_ids)}
movie_to_idx = {mid: idx for idx, mid in enumerate(movie_ids)}
idx_to_movie = {idx: mid for mid, idx in movie_to_idx.items()}

user_indices = train_df["userId"].map(user_to_idx).values
movie_indices = train_df["movieId"].map(movie_to_idx).values
ratings = train_df["rating"].values.astype(np.float32)

print(f"Users: {len(user_ids):,} | Movies: {len(movie_ids):,} | Ratings: {len(ratings):,}")

Users: 138,493 | Movies: 26,707 | Ratings: 19,861,770


In [3]:
item_user_matrix = csr_matrix(
    (ratings, (movie_indices, user_indices)),
    shape=(len(movie_ids), len(user_ids))
)

memory_mb = (
    item_user_matrix.data.nbytes
    + item_user_matrix.indices.nbytes
    + item_user_matrix.indptr.nbytes
) / (1024 ** 2)

print("Matrix shape (movies x users):", item_user_matrix.shape)
print(f"Sparse matrix memory usage: {memory_mb:.2f} MB")

Matrix shape (movies x users): (26707, 138493)
Sparse matrix memory usage: 151.64 MB


In [5]:
model_knn = NearestNeighbors(metric="cosine", algorithm="brute", n_neighbors=11, n_jobs=-1)
model_knn.fit(item_user_matrix)

movies = pd.read_csv(MOVIES_PATH)

def get_similar_movies(movie_id, n=10):
    if movie_id not in movie_to_idx:
        return "Movie not found in training data."
    
    idx = movie_to_idx[movie_id]
    distances, indices = model_knn.kneighbors(item_user_matrix[idx], n_neighbors=n + 1)
    
    similar_ids = [idx_to_movie[i] for i in indices.flatten()[1:]]
    scores = [round(1 - d, 4) for d in distances.flatten()[1:]]
    
    result = pd.DataFrame({"movieId": similar_ids, "similarity": scores})
    return result.merge(movies, on="movieId", how="left")

# Test with The Matrix (movieId = 2571)
get_similar_movies(858)

,movieId,similarity,title,genres
0,1221,0.7622,"Godfather: Part II, The (1974)",Crime|Drama
1,1213,0.5808,Goodfellas (1990),Crime|Drama
2,1198,0.5626,Raiders of the Lost Ark (Indiana Jones and the...,Action|Adventure
3,260,0.5596,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi
4,1193,0.5589,One Flew Over the Cuckoo's Nest (1975),Drama
5,1196,0.5435,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Sci-Fi
6,1208,0.5285,Apocalypse Now (1979),Action|Drama|War
7,608,0.5271,Fargo (1996),Comedy|Crime|Drama|Thriller
8,2571,0.5204,"Matrix, The (1999)",Action|Sci-Fi|Thriller
9,1089,0.5184,Reservoir Dogs (1992),Crime|Mystery|Thriller


In [6]:
def recommend_for_user(user_id, n=10, top_liked_limit=5):
    user_data = train_df[train_df["userId"] == user_id]
    
    if user_data.empty:
        return "New user (Cold Start) - use Popularity Baseline."
    
    watched_movies = set(user_data["movieId"])
    
    # Get user's highest-rated movies (4.0 and above)
    liked_movies = (
        user_data[user_data["rating"] >= 4.0]
        .sort_values(by="rating", ascending=False)
        .head(top_liked_limit)
    )
    
    if liked_movies.empty:
        return "User has no highly rated movies - use Popularity Baseline."
    
    candidate_scores = {}
    
    for _, row in liked_movies.iterrows():
        m_id = row["movieId"]
        user_rating = row["rating"]
        
        if m_id not in movie_to_idx:
            continue
            
        idx = movie_to_idx[m_id]
        distances, indices = model_knn.kneighbors(item_user_matrix[idx], n_neighbors=20)
        
        for dist, neighbor_idx in zip(distances.flatten()[1:], indices.flatten()[1:]):
            neighbor_id = idx_to_movie[neighbor_idx]
            if neighbor_id in watched_movies:
                continue
            
            similarity = 1 - dist
            # Weight similarity by how much the user liked the source movie
            candidate_scores[neighbor_id] = candidate_scores.get(neighbor_id, 0) + (similarity * user_rating)
            
    # Sort candidates by total score
    top_candidates = sorted(candidate_scores.items(), key=lambda x: x[1], reverse=True)[:n]
    
    rec_df = pd.DataFrame(top_candidates, columns=["movieId", "score"])
    rec_df["score"] = rec_df["score"].round(3)
    return rec_df.merge(movies, on="movieId", how="left")

In [7]:
test_user = 1

print(f"--- Movies User {test_user} Liked (Sample) ---")
user_history = (
    train_df[(train_df["userId"] == test_user) & (train_df["rating"] >= 4.0)]
    .sort_values(by="rating", ascending=False)
    .head(5)
    .merge(movies, on="movieId", how="left")
)
display(user_history[["movieId", "rating", "title", "genres"]])

print(f"\n--- Top 10 Recommendations for User {test_user} ---")
display(recommend_for_user(test_user, n=10))

--- Movies User 1 Liked (Sample) ---


,movieId,rating,title,genres
0,7153,5.0,"Lord of the Rings: The Return of the King, The...",Action|Adventure|Drama|Fantasy
1,5952,5.0,"Lord of the Rings: The Two Towers, The (2002)",Adventure|Fantasy
2,4993,5.0,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy
3,8507,5.0,Freaks (1932),Crime|Drama|Horror
4,1198,4.5,Raiders of the Lost Ark (Indiana Jones and the...,Action|Adventure



--- Top 10 Recommendations for User 1 ---


,movieId,score,title,genres
0,2571,12.333,"Matrix, The (1999)",Action|Sci-Fi|Thriller
1,5349,9.167,Spider-Man (2002),Action|Adventure|Sci-Fi|Thriller
2,6377,9.012,Finding Nemo (2003),Adventure|Animation|Children|Comedy
3,5445,8.846,Minority Report (2002),Action|Crime|Mystery|Sci-Fi|Thriller
4,3578,8.763,Gladiator (2000),Action|Adventure|Drama
5,4963,8.709,Ocean's Eleven (2001),Crime|Thriller
6,6874,8.670,Kill Bill: Vol. 1 (2003),Action|Crime|Thriller
7,4886,8.610,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy
8,5418,8.460,"Bourne Identity, The (2002)",Action|Mystery|Thriller
9,33794,5.804,Batman Begins (2005),Action|Crime|IMAX


In [8]:
from sklearn.decomposition import TruncatedSVD

# Transpose our existing item_user_matrix to (users x movies)
user_item_matrix = item_user_matrix.T.tocsr()

svd = TruncatedSVD(n_components=50, random_state=42)
user_factors = svd.fit_transform(user_item_matrix).astype(np.float32)
item_factors = svd.components_.T.astype(np.float32)

print("User factors shape:", user_factors.shape)
print("Item factors shape:", item_factors.shape)

User factors shape: (138493, 50)
Item factors shape: (26707, 50)


In [9]:
def recommend_svd(user_id, n=10):
    if user_id not in user_to_idx:
        return "New user (Cold Start) - use Popularity Baseline."
    
    u_idx = user_to_idx[user_id]
    
    # Predict scores for all 26,707 movies at once
    scores = user_factors[u_idx] @ item_factors.T
    
    # Hide movies the user has already watched in training
    watched_indices = user_item_matrix[u_idx].indices
    scores[watched_indices] = -np.inf
    
    # Get top N movie indices
    top_indices = np.argpartition(scores, -n)[-n:]
    top_indices = top_indices[np.argsort(scores[top_indices])[::-1]]
    
    rec_df = pd.DataFrame({
        "movieId": [idx_to_movie[i] for i in top_indices],
        "svd_score": np.round(scores[top_indices], 3)
    })
    return rec_df.merge(movies, on="movieId", how="left")

# Test on User 1
recommend_svd(1, n=10)

,movieId,svd_score,title,genres
0,1197,3.630,"Princess Bride, The (1987)",Action|Adventure|Comedy|Fantasy|Romance
1,2571,2.501,"Matrix, The (1999)",Action|Sci-Fi|Thriller
2,2115,2.255,Indiana Jones and the Temple of Doom (1984),Action|Adventure|Fantasy
3,1206,2.133,"Clockwork Orange, A (1971)",Crime|Drama|Sci-Fi|Thriller
4,1073,2.119,Willy Wonka & the Chocolate Factory (1971),Children|Comedy|Fantasy|Musical
5,3793,1.968,X-Men (2000),Action|Adventure|Sci-Fi
6,5618,1.898,Spirited Away (Sen to Chihiro no kamikakushi) ...,Adventure|Animation|Fantasy
7,551,1.851,"Nightmare Before Christmas, The (1993)",Animation|Children|Fantasy|Musical
8,1275,1.835,Highlander (1986),Action|Adventure|Fantasy
9,6874,1.812,Kill Bill: Vol. 1 (2003),Action|Crime|Thriller


In [10]:
TEST_PATH = PROJECT_ROOT / "data" / "processed" / "splits" / "test.parquet"
test_df = pd.read_parquet(TEST_PATH, columns=["userId", "movieId", "rating"])

# Keep only relevant test ratings (>= 4.0) for users & movies seen in training
relevant_test = test_df[
    (test_df["rating"] >= 4.0) &
    (test_df["userId"].isin(user_to_idx)) &
    (test_df["movieId"].isin(movie_to_idx))
]

# Sample 5,000 test interactions for fast evaluation
sample_test = relevant_test.sample(n=5000, random_state=42)

hits = 0
for row in sample_test.itertuples(index=False):
    u_idx = user_to_idx[row.userId]
    target_m_idx = movie_to_idx[row.movieId]
    
    scores = user_factors[u_idx] @ item_factors.T
    scores[user_item_matrix[u_idx].indices] = -np.inf
    
    top_10_idx = np.argpartition(scores, -10)[-10:]
    if target_m_idx in top_10_idx:
        hits += 1

svd_recall_at_10 = hits / len(sample_test)
print(f"Evaluated on: {len(sample_test)} relevant test ratings")
print(f"Hits: {hits}")
print(f"SVD Recall@10: {svd_recall_at_10:.4%}")

Evaluated on: 5000 relevant test ratings
Hits: 681
SVD Recall@10: 13.6200%


In [11]:
import joblib

MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

np.save(MODEL_DIR / "svd_user_factors.npy", user_factors)
np.save(MODEL_DIR / "svd_item_factors.npy", item_factors)

joblib.dump({
    "user_to_idx": user_to_idx,
    "movie_to_idx": movie_to_idx,
    "idx_to_movie": idx_to_movie
}, MODEL_DIR / "svd_mappings.joblib")

print("SVD model artifacts saved successfully!")

SVD model artifacts saved successfully!
